# 19 — Population calibration against published [177Lu]Lu-PSMA-617 half-lives and doses: plan (b)

**The population of the frozen cohort is in §17-§18. §13-§16: first cohort. §9-§12: the agent factor, before the spread targets. §1-§8: first calibration without it.**

**Aim.** At population level: $\lambda_{rel}$ per structure, therapy-ligand $K_D$ and $\lambda_{int}$, the lesion $[R]$ population. **Half-lives have priority over doses; the dose residual measures the agent mismatch and is not fitted away.** Identifiability first. **Training split only. No cohort is generated here.**

**What comes from the real PET (nb18, training split):** per-patient organ densities, $f_{SAL}$, GFR, hematocrit under the PET ligand, and for lesions only delivery (flow) and the lower bound on density, through the censored likelihood. Lesion density population: the five-pair start, its median calibrated here.

**Dose per TIA:** self-dose only, S-values mass-scaled as in nb14 (EANM 2022 Tables 2 and 5, IDAC-Dose 2.1): kidneys 0.204 at 422 g, liver 0.0376 at 2360 g, spleen 0.377 at 228.4 g, salivary 0.947 at 88.98 g, mGy/(MBq h), times $m_{ref}/m$. Lesions: local deposition 0.0853 mJ/(MBq h), which equals Table C of Kletting within 2% at 0.5-1 mL. Monte Carlo later. Therapy: 7.4 GBq, 100 nmol (amount assumed).

**About this notebook.** Calibration of the therapy population (release rates, agent factors, spreads) against published PSMA-617 half-lives and doses.
- **Inputs:** nb18 outputs, `cohort_v1/` (§17).
- **Outputs:** `data/processed/psma/pbpk/nb19_*` (`nb19_cohort_population_v2.pkl`).
- **Run time:** about 15 min with the caches; about 1.2 h from scratch.
- **Environment:** .venv.

In [ ]:
import os
os.environ.update(OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")
import json
import multiprocessing as mp
import pickle
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import RBFInterpolator
from scipy.optimize import minimize

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "paths.py").exists())
sys.path.insert(0, str(ROOT / "src"))
import pbpk as P
import pbpk_calibration as C
import pbpk_fit as F

warnings.filterwarnings("ignore", category=UserWarning)
OUT = ROOT / "data/processed/psma/pbpk"
N_PROCESSES = 5
SEED = 20261106
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
INK, MUTED = "#0b0b0b", "#52514e"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e6e3", "grid.linewidth": 0.6, "axes.axisbelow": True, "font.size": 9,
                     "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED, "axes.edgecolor": "#b8b7ae"})
pd.set_option("display.width", 230, "display.max_columns", 40, "display.max_rows", 150)


def cached(name, build):
    path = OUT / f"nb19_{name}.pkl"
    if path.exists():
        return pickle.loads(path.read_bytes())
    value = build()
    path.write_bytes(pickle.dumps(value))
    return value

## 1. Targets: what was read

| Source | Read | Population, method | Gives |
|---|---|---|---|
| Delker 2016 | **abstract only**: paywalled, not in PMC | 5 pts, 3.6 GBq, planar + SPECT at 1-72 h | doses; **no half-life in the abstract** |
| Violet 2019 | full text of the journal page **without table cells** | 30 pts mCRPC, SPECT at 4, 24, 96 h, voxel | doses in the abstract; **no half-life in the text** (3-phase model, values not given) |
| Rosar 2021 | full text (PMC8099965) | 24 pts mCRPC, 2D, hybrid, 3D | doses, Table 2; **no half-life** |
| Ells 2024 | Table 3 and text (PMC11294071) | meta-analysis, 617: kidney 17 studies, 297 pts | pooled doses; **no half-life** |
| Ells 2026 | full text (PMC13635081) | RESIST-PC, n 38-48, planar and SPECT at 4-96 h | doses |
| Kurth 2018 | results (PMC5897276) | whole body, external probe | $T_{eff}$ 1.7 ± 0.8 h and 41.1 ± 9.3 h |
| Schütze 2026 | results (PMC13046430) | Pluvicto, dose rate, n = 55 datasets | $T_{eff}$ < 168 h: 1.60 ± 0.63 d, median 1.41 d |
| **Added:** Sarnelli 2019 | full text (PMC6385027) | 9 pts mCRPC, planar 1-120 h, with mannitol and glutamate | organ and whole-body $T_{eff}$, doses |
| **Added:** Peters 2023 | full text (PMC9873880) | 10 pts **low-volume mHSPC**, SPECT 5 points | organ and lesion $T_{eff}$, doses |
| **Added:** Peters 2022a, 2022b | full text (PMC8803803, PMC8921092) | **same centre and trial as Peters 2023**, so not independent | two-phase kidney and liver $T_{eff}$; kidney dose predicted from PSMA-11 PET over SPECT 2.21 ± 0.46 |

**The three sources named for organ half-lives do not give them in what could be read.** Delker's tables may do, behind the paywall. So organ and lesion half-lives come from Sarnelli and Peters. **No spleen half-life was found:** the spleen is tied to the liver. **Lesion half-lives come from one low-volume hormone-sensitive cohort with small lesions, not from mCRPC.** Violet's abstract prints Gy/MBq; the text gives Gy/GBq.

**Tolerances.** Log SD between studies. For half-lives there are 2-3 studies, so a floor of 0.10 is added in quadrature (assumption, planar against SPECT is not sampled). Target for doses: geometric mean of the four sources named (Ells 2024, Ells 2026, Rosar, Violet); the tolerance includes also Delker, Sarnelli, Peters. Ells 2024 pools earlier studies, so these are not independent, and the tolerance is if anything small. Salivary glands: geometric mean of parotid and submandibular within a study. Lesions: bone and soft tissue likewise; Violet's value is the whole-body tumour dose, median 11.55 Gy at a mean 7.8 GBq.

In [ ]:
HALF_LIVES = {
    "teff_kidneys": dict(Sarnelli2019=31.4, Peters2023=39.0),
    "teff_liver": dict(Sarnelli2019=25.4, Peters2023=29.0),
    "teff_salivary": dict(Sarnelli2019=33.0, Peters2023=33.0),
    "teff_lesions": dict(Peters2023=69.0),
    "teff_whole_body": dict(Kurth2018=41.1, Schuetze2026=1.60 * 24, Sarnelli2019=40.1),
}
gm = lambda *v: float(np.exp(np.mean(np.log(v))))
DOSES = {
    "dose_kidneys": dict(Ells2024=0.58, Ells2026=0.29, Rosar2021=0.54, Violet2019=0.39, Delker2016=0.6, Sarnelli2019=0.70, Peters2023=0.52),
    "dose_liver": dict(Ells2024=0.16, Ells2026=0.11, Rosar2021=0.10, Violet2019=0.10, Delker2016=0.1, Sarnelli2019=0.13, Peters2023=0.08),
    "dose_spleen": dict(Violet2019=0.06, Delker2016=0.1),
    "dose_salivary": dict(Ells2024=gm(0.84, 0.74), Ells2026=gm(0.26, 0.23), Rosar2021=gm(0.81, 0.72), Violet2019=gm(0.58, 0.44), Delker2016=1.4, Sarnelli2019=0.48, Peters2023=0.50),
    "dose_lesions": dict(Ells2024=gm(3.57, 4.19), Ells2026=gm(1.74, 3.21), Rosar2021=1.68, Violet2019=11.55 / 7.8, Delker2016=13.1, Peters2023=2.07),
}
NAMED = ("Ells2024", "Ells2026", "Rosar2021", "Violet2019")
FLOOR = 0.10
rows = []
for key, values in HALF_LIVES.items():
    logs = np.log(list(values.values()))
    between = logs.std(ddof=1) if len(logs) > 1 else 0.0
    rows.append(dict(target=key, value=float(np.exp(logs.mean())), tolerance=float(np.hypot(between, FLOOR)), between_study=between, n_sources=len(logs), sources=", ".join(values)))
for key, values in DOSES.items():
    named = [v for k, v in values.items() if k in NAMED]
    logs = np.log(list(values.values()))
    rows.append(dict(target=key, value=float(np.exp(np.mean(np.log(named)))), tolerance=float(logs.std(ddof=1)), between_study=float(logs.std(ddof=1)), n_sources=len(logs), sources=", ".join(values)))
targets = pd.DataFrame(rows).set_index("target").loc[list(C.OUTPUTS)]
targets.to_csv(OUT / "nb19_targets.csv")
display(targets.round(3))
Y = np.log(targets.value.to_numpy())
TOL = targets.tolerance.to_numpy()
T_ONLY = np.array([k.startswith("teff") for k in C.OUTPUTS])

## 2. Patients and the base model

One study per subject of the training split, every second subject, from the final pass of nb18: 66 patients with their lesions. Base: Kletting's $\lambda_{rel}$, $K_D$ 1 nM, $\lambda_{int}$ 1e-3 /min, lesion median 136. **Model over target at base, before anything is calibrated.**

In [ ]:
eb_real = pickle.loads((OUT / "nb18_eb_real.pkl").read_bytes())
results = eb_real["fits"][-1]["likelihood_results"]
split = json.loads((ROOT / "data/processed/psma/psma_68ga_split.json").read_text())
assert not set(split["test_studies"]) & {r["study_id"] for r in results}
first = {}
for r in results:
    first.setdefault(r["subject"], r)
chosen = list(first.values())[::2]


def make_base(r):
    q = r["row"]
    return C.Base(F.load_geometry(r["study_id"]), dict(kidneys=q["r_kidneys"], liver=q["r_liver"], spleen=q["r_spleen"], salivary=F.PRIORS["salivary"][0]), q["f_sal"], q["gfr"], q["hematocrit"], r["likelihood"], F.hash_id(r["study_id"]))


bases = [make_base(r) for r in chosen]
del eb_real, results, first, chosen
print(len(bases), "patients;", sum(len(b.geometry.lesions_ml) for b in bases), "lesions;", sum(int((b.geometry.lesions_ml >= C.MIN_LESION_ML).sum()) for b in bases), "at or above", C.MIN_LESION_ML, "mL")


def one(job):
    index, theta = job
    return C.forward(bases[index], theta)


def evaluate_many(thetas, workers):
    jobs = [(i, tuple(t)) for t in thetas for i in range(len(bases))]
    output = workers.map(one, jobs, chunksize=len(bases))
    out = np.zeros((len(thetas), len(C.OUTPUTS)))
    for k in range(len(thetas)):
        pop = C.population(output[k * len(bases): (k + 1) * len(bases)])
        out[k] = [np.log(pop[n]) for n in C.OUTPUTS]
    return out


def evaluate(thetas):
    with mp.get_context("fork").Pool(N_PROCESSES) as workers:
        return evaluate_many(np.atleast_2d(thetas), workers)


THETA0 = np.array([C.BASE[k] for k in C.NAMES])
base_out = cached("base", lambda: evaluate(THETA0))[0]
report = pd.DataFrame(dict(model=np.exp(base_out), target=targets.value.to_numpy(), tolerance=TOL, model_over_target=np.exp(base_out - Y), pull=(base_out - Y) / TOL), index=C.OUTPUTS)
report.to_csv(OUT / "nb19_base_against_targets.csv")
display(report.round(3))

## 3. Emulator

The forward model is direct (66 patients in about 2 s), but identifiability needs many fits. Thin-plate emulator of the 10 log outputs on 7 parameters: 1000 points over the whole box, 1500 focused where half-lives are plausible ($\lambda_{rel}$ 5e-5 to 1e-3 /min). 120 further points held out in the focused region are the check of accuracy and the truths of §4. All design points are direct simulations with the same random numbers per patient.

In [ ]:
FOCUS = C.BOX.copy()
FOCUS[2:6] = np.log([[5e-5, 1e-3]] * 4)


def design_build():
    rng = np.random.default_rng(SEED)
    wide, focus, hold = C.latin_hypercube(1000, rng), C.latin_hypercube(1500, rng, FOCUS), C.latin_hypercube(120, rng, FOCUS)
    with mp.get_context("fork").Pool(N_PROCESSES) as workers:
        return dict(x=np.vstack([wide, focus]), y=evaluate_many(np.vstack([wide, focus]), workers), hold_x=hold, hold_y=evaluate_many(hold, workers))


design = cached("design", design_build)
LOW, HIGH = C.BOX[:, 0], C.BOX[:, 1]
unit = lambda x: (x - LOW) / (HIGH - LOW)
emulator = RBFInterpolator(unit(design["x"]), design["y"], kernel="thin_plate_spline", smoothing=1e-4)
holdout = emulator(unit(design["hold_x"])) - design["hold_y"]
accuracy = pd.DataFrame(dict(rms=np.sqrt((holdout**2).mean(axis=0)), p95=np.quantile(np.abs(holdout), 0.95, axis=0), tolerance=TOL), index=C.OUTPUTS)
accuracy["rms_over_tolerance"] = accuracy.rms / accuracy.tolerance
accuracy.to_csv(OUT / "nb19_emulator_accuracy.csv")
display(accuracy.round(3))

## 4. Identifiability, before any final fit

Three checks.

1. **Local:** sensitivities $\partial\log y/\partial\log\theta$ at the base, direct simulation, and the singular values of the matrix scaled by the tolerances.
2. **Simulation-based recovery:** the 120 held-out populations are truths, each observed through its direct outputs with noise at the tolerances. The ensemble sampler on the emulator, flat in the log box, gives a posterior. Asked for each parameter: prior SD over posterior SD, error of the posterior median, coverage of the 90% interval. Target sets: half-lives only, doses only, both.
3. **Profile on the real targets** (§5).

**Rule, set here:** a parameter is separable if its recovery error, in log, is below half its focused-box SD and the posterior SD is below a third of it. Otherwise it is reported as not separable and its value is not adopted.

In [ ]:
def sensitivities():
    steps = np.log(1.5)
    with mp.get_context("fork").Pool(N_PROCESSES) as workers:
        up = evaluate_many(THETA0 + steps * np.eye(len(THETA0)), workers)
        down = evaluate_many(THETA0 - steps * np.eye(len(THETA0)), workers)
    return (up - down) / (2 * steps)


J = cached("sensitivity", sensitivities).T
sens = pd.DataFrame(J, index=C.OUTPUTS, columns=C.NAMES)
sens.to_csv(OUT / "nb19_sensitivity.csv")
display(sens.round(2))
scaled = J / TOL[:, None]
values = np.linalg.svd(scaled, compute_uv=False)
_, _, right = np.linalg.svd(scaled)
print("Singular values, all targets:", np.round(values, 2))
print("Half-lives only:", np.round(np.linalg.svd(scaled[T_ONLY], compute_uv=False), 2), "| doses only:", np.round(np.linalg.svd(scaled[~T_ONLY], compute_uv=False), 2))
print("Direction of the smallest singular value:", dict(zip(C.NAMES, np.round(right[-1], 2))))


def log_posterior(points, observed, mask):
    inside = np.all((points >= LOW) & (points <= HIGH), axis=1)
    out = np.full(len(points), -np.inf)
    if inside.any():
        z = (emulator(unit(points[inside])) - observed)[:, mask] / TOL[mask]
        out[inside] = -0.5 * np.sum(z**2, axis=1)
    return out


def sample(observed, mask, seed, n_steps=700, walkers=64):
    rng = np.random.default_rng(seed)
    start = C.latin_hypercube(walkers * 40, rng, FOCUS)
    lp = log_posterior(start, observed, mask)
    start = start[np.argsort(lp)[-walkers:]] + rng.normal(0, 0.05, (walkers, len(LOW)))
    start = np.clip(start, LOW + 1e-3, HIGH - 1e-3)
    chain, acceptance = C.stretch_sampler(lambda p: log_posterior(p, observed, mask), start, n_steps, rng)
    return chain[n_steps // 2 :].reshape(-1, len(LOW))[::5], acceptance


SETS = {"half-lives": T_ONLY, "doses": ~T_ONLY, "both": np.ones(len(TOL), bool)}


def recovery_job(job):
    index, name, seed = job
    rng = np.random.default_rng(seed)
    observed = design["hold_y"][index] + rng.normal(0, TOL)
    draws, acceptance = sample(observed, SETS[name], seed)
    truth = design["hold_x"][index]
    q = np.quantile(draws, [0.05, 0.5, 0.95], axis=0)
    return dict(index=index, name=name, truth=truth, median=q[1], sd=draws.std(axis=0), inside=(q[0] <= truth) & (truth <= q[2]), acceptance=acceptance)


def recovery_build():
    jobs = [(i, name, SEED + 1000 * i + k) for i in range(120) for k, name in enumerate(SETS)]
    with mp.get_context("fork").Pool(N_PROCESSES) as workers:
        return workers.map(recovery_job, jobs, chunksize=2)


recovery = cached("recovery", recovery_build)
focus_sd = (FOCUS[:, 1] - FOCUS[:, 0]) / np.sqrt(12)
rows = []
for name in SETS:
    sub = [r for r in recovery if r["name"] == name]
    error = np.array([r["median"] - r["truth"] for r in sub])
    sd = np.array([r["sd"] for r in sub])
    inside = np.array([r["inside"] for r in sub])
    for i, p in enumerate(C.NAMES):
        rms = float(np.sqrt(np.mean(error[:, i] ** 2)))
        rows.append(dict(targets=name, parameter=p, box_sd=focus_sd[i], rms_error=rms, posterior_sd=float(np.median(sd[:, i])), coverage_90=float(inside[:, i].mean()), separable=bool(rms < 0.5 * focus_sd[i] and np.median(sd[:, i]) < focus_sd[i] / 3)))
identifiability = pd.DataFrame(rows)
identifiability.to_csv(OUT / "nb19_identifiability.csv", index=False)
display(identifiability.pivot(index="parameter", columns="targets", values=["rms_error", "posterior_sd", "coverage_90", "separable"]).loc[list(C.NAMES)].round(2))
print("Box SD:", dict(zip(C.NAMES, np.round(focus_sd, 2))), "| acceptance median", round(np.median([r["acceptance"] for r in recovery]), 2))

## 5. Calibration on the published targets

Three arms, each checked by direct simulation.

- **A, half-lives only:** $\lambda_{rel}$ of the four structures from the half-lives, ligand and lesion median at base. **The dose residual here is the agent mismatch at the ligand of Kletting.**
- **B, A plus lesion median:** the lesion median from the lesion dose, with the half-lives of A.
- **C, all parameters, half-lives and doses.** What the ligand can absorb. The half-lives are the tighter terms, so they have priority through their tolerances; if the doses drag them outside, it shows in the table.

Then the profile of each parameter in arm C, with the emulator.

In [ ]:
real_chain, real_acceptance = cached("posterior_real", lambda: sample(Y, SETS["both"], SEED, n_steps=2500, walkers=128))
half_chain, _ = cached("posterior_half", lambda: sample(Y, SETS["half-lives"], SEED + 1, n_steps=2500, walkers=128))
print("Acceptance", round(real_acceptance, 2))
rel = [2, 3, 4, 5]


def fit_a():
    def nll(x):
        t = THETA0.copy()
        t[rel] = x
        z = (emulator(unit(t[None]))[0] - Y)[T_ONLY] / TOL[T_ONLY]
        return 0.5 * np.sum(z**2)
    return minimize(nll, THETA0[rel], method="Nelder-Mead", options=dict(maxiter=2000, xatol=1e-3, fatol=1e-4)).x


theta_a = THETA0.copy()
theta_a[rel] = fit_a()


def fit_b():
    def nll(x):
        t = theta_a.copy()
        t[6] = x[0]
        return 0.5 * ((emulator(unit(t[None]))[0][C.OUTPUTS.index("dose_lesions")] - Y[C.OUTPUTS.index("dose_lesions")]) / TOL[C.OUTPUTS.index("dose_lesions")]) ** 2
    best = min((minimize(nll, [s], method="Nelder-Mead") for s in np.log([50.0, 200.0, 800.0])), key=lambda r: r.fun)
    return best.x[0]


theta_b = theta_a.copy()
theta_b[6] = np.clip(fit_b(), LOW[6], HIGH[6])
start_c = real_chain[np.argmax(log_posterior(real_chain, Y, SETS["both"]))]
theta_c = minimize(lambda x: -log_posterior(x[None], Y, SETS["both"])[0] if np.all((x >= LOW) & (x <= HIGH)) else 1e6, start_c, method="Nelder-Mead", options=dict(maxiter=4000, xatol=1e-3, fatol=1e-3)).x
direct = cached("arms_direct", lambda: evaluate(np.vstack([theta_a, theta_b, theta_c])))
arms = pd.DataFrame({"target": targets.value, "tolerance": TOL, "base": np.exp(base_out), "A": np.exp(direct[0]), "B": np.exp(direct[1]), "C": np.exp(direct[2])}, index=C.OUTPUTS)
for name, y in [("base", base_out), ("A", direct[0]), ("B", direct[1]), ("C", direct[2])]:
    arms[f"{name} ratio"] = np.exp(y - Y)
arms.to_csv(OUT / "nb19_arms.csv")
display(arms.round(3))
print("Chi2 of half-lives and doses, direct simulation")
display(pd.DataFrame({name: dict(half_lives=float(np.sum(((y - Y)[T_ONLY] / TOL[T_ONLY]) ** 2)), doses=float(np.sum(((y - Y)[~T_ONLY] / TOL[~T_ONLY]) ** 2))) for name, y in [("base", base_out), ("A", direct[0]), ("B", direct[1]), ("C", direct[2])]}).round(1))
q = np.quantile(real_chain, [0.05, 0.5, 0.95], axis=0)
posterior = pd.DataFrame(dict(base=np.exp(THETA0), A=np.exp(theta_a), B=np.exp(theta_b), C_map=np.exp(theta_c), C_q05=np.exp(q[0]), C_q50=np.exp(q[1]), C_q95=np.exp(q[2]), half_lives_q05=np.exp(np.quantile(half_chain, 0.05, axis=0)), half_lives_q95=np.exp(np.quantile(half_chain, 0.95, axis=0))), index=C.NAMES)
posterior.to_csv(OUT / "nb19_parameters.csv")
display(posterior.map(lambda v: float(f"{v:.3g}")))
correlation = pd.DataFrame(np.corrcoef(real_chain.T), index=C.NAMES, columns=C.NAMES)
correlation.to_csv(OUT / "nb19_posterior_correlation.csv")
display(correlation.round(2))

In [ ]:
def profile_job(job):
    index, value, name = job
    rng = np.random.default_rng(int(SEED + index * 100 + abs(value) * 1000))
    mask = SETS[name]
    box = FOCUS.copy()
    box[index] = (value, value)
    points = C.latin_hypercube(3000, rng, box)
    points[:, index] = value
    lp = log_posterior(points, Y, mask)
    best = -np.inf
    free = [i for i in range(len(LOW)) if i != index]
    for start in points[np.argsort(lp)[-3:]]:
        def nll(x):
            t = start.copy()
            t[free] = x
            return -log_posterior(t[None], Y, mask)[0] if np.all((t >= LOW) & (t <= HIGH)) else 1e6
        best = max(best, -minimize(nll, start[free], method="Nelder-Mead", options=dict(maxiter=900, xatol=2e-3, fatol=2e-3)).fun)
    return dict(parameter=C.NAMES[index], value=value, targets=name, chi2=-2 * best)


def profiles_build():
    jobs = [(i, v, name) for name in ("both", "half-lives") for i in range(len(LOW)) for v in np.linspace(LOW[i], HIGH[i], 15)]
    with mp.get_context("fork").Pool(N_PROCESSES) as workers:
        return workers.map(profile_job, jobs, chunksize=2)


profiles = pd.DataFrame(cached("profiles", profiles_build))
profiles["delta"] = profiles.chi2 - profiles.groupby(["parameter", "targets"]).chi2.transform("min")
profiles.to_csv(OUT / "nb19_profiles.csv", index=False)
fig, axes = plt.subplots(2, 4, figsize=(10, 4.4), constrained_layout=True)
for ax, name in zip(axes.ravel(), C.NAMES):
    for (kind, sub), color in zip(profiles[profiles.parameter == name].groupby("targets"), COLORS):
        ax.plot(np.exp(sub.value), sub.delta, color=color, lw=1.6, label=kind)
    ax.axhline(3.84, color=MUTED, lw=0.8, ls=":")
    ax.set(xscale="log", ylim=(0, 25), title=name.replace("log_", ""))
axes[0, 0].set_ylabel(r"Profile $\Delta\chi^2$")
axes[1, 0].set_ylabel(r"Profile $\Delta\chi^2$")
axes[0, 0].legend(frameon=False, fontsize=7)
axes[1, 3].axis("off")
fig.savefig(OUT / "nb19_profiles.png", dpi=160)
inside = profiles[profiles.delta <= 3.84].groupby(["parameter", "targets"]).value.agg(["min", "max"]).apply(np.exp).unstack(1).loc[list(C.NAMES)]
inside.to_csv(OUT / "nb19_profile_intervals.csv")
display(inside.map(lambda v: float(f"{v:.3g}")))
display(profiles.groupby("targets").chi2.min().round(1).to_frame("best chi2"))

## 6. Between-patient spread of $\lambda_{rel}$

$T_{eff}\approx\ln2/(\lambda_{phys}+\lambda_{rel})$ (nb12, checked below on the calibrated model), so a log SD of $T_{eff}$ maps to one of $\lambda_{rel}$: $\sigma_{\log\lambda_{rel}}=\sigma_{\log T}\,(\lambda_{phys}+\lambda_{rel})/\lambda_{rel}$. Published SDs include measurement error, so these are **upper limits**. Then a check: draw from these SDs per patient and compare the $T_{eff}$ spread obtained.

In [ ]:
LAMBDA_PHYS = np.log(2) / (6.647 * 24 * 60)
SPREADS = {"kidneys": dict(Peters2023=5 / 39, Sarnelli2019_range=np.log(80.6 / 12.2) / 3.92), "liver": dict(Peters2023=8 / 29, Sarnelli2019_range=np.log(62.9 / 12.5) / 3.92), "salivary": dict(Peters2023=4 / 33, Sarnelli2019_range=np.log(60.7 / 25.6) / 3.92), "lesions": dict(Peters2023=13 / 69, Peters2022_range=np.log(137 / 51) / 3.92)}
index = dict(kidneys=2, liver=3, salivary=4, lesions=5)
rows = []
for key, values in SPREADS.items():
    rate = float(np.exp(theta_a[index[key]]))
    factor = (LAMBDA_PHYS + rate) / rate
    implied_T = np.log(2) / (LAMBDA_PHYS + rate) / 60
    model_T = float(np.exp(direct[0][C.OUTPUTS.index(f"teff_{key}")]))
    sd = float(np.mean(list(values.values())))
    rows.append(dict(structure=key, lambda_rel_per_min=rate, closed_form_T_h=implied_T, model_T_h=model_T, log_sd_T_small=min(values.values()), log_sd_T_large=max(values.values()), log_sd_lambda_low=min(values.values()) * factor, log_sd_lambda_high=max(values.values()) * factor, kletting_median=dict(kidneys=F.PRIORS["l_kidney"][0], liver=F.PRIORS["l_kidney"][0], salivary=F.PRIORS["l_salivary"][0], lesions=F.PRIORS["l_tumor"][0])[key], kletting_log_sd=dict(kidneys=F.PRIORS["l_kidney"][1], liver=F.PRIORS["l_kidney"][1], salivary=F.PRIORS["l_salivary"][1], lesions=F.PRIORS["l_tumor"][1])[key]))
spreads = pd.DataFrame(rows).set_index("structure")
spreads.to_csv(OUT / "nb19_lambda_rel.csv")
display(spreads.map(lambda v: float(f"{v:.3g}")))


def spread_job(job):
    i, seed, sds = job
    rng = np.random.default_rng(seed)
    t = theta_a.copy()
    for k, s in zip(rel, sds):
        t[k] = t[k] + rng.normal(0, s)
    return C.forward(bases[i], tuple(t))


def spread_build():
    sds = [spreads.loc[k, "log_sd_lambda_low"] for k in ("kidneys", "liver", "salivary", "lesions")]
    with mp.get_context("fork").Pool(N_PROCESSES) as workers:
        return workers.map(spread_job, [(i, SEED + i, sds) for i in range(len(bases))], chunksize=4)


rows_drawn = cached("spread_check", spread_build)
display(pd.DataFrame({k: dict(median=np.median([r[k] for r in rows_drawn if k in r]), log_sd=np.log([r[k] for r in rows_drawn if k in r]).std()) for k in C.OUTPUTS}).T.round(3))

## 7. Where the dose residual comes from

Two checks without any fit. **(a) Arithmetic, no model.** Median 1 h kidney and liver %IA of the real PET times $T_{eff}/\ln2$ of the targets, through the same S-value. The result assumes the 1 h uptake is retained and is an upper limit. **(b) Saturation in the model.** Arm A at 100 nmol against 2 nmol, the same activity. The fitted organ densities are low in liver (about 1.3 nmol/L), so the therapy amount can saturate them; whatever the real liver holds that is not receptor-bound does not saturate, and the model has no such term.

In [ ]:
masses = {k: np.median([getattr(b.geometry, f"{k}_ml") for b in bases]) for k in ("kidneys", "liver", "spleen")}
rows_real = pd.read_csv(OUT / "nb18_real_after_eb.csv")
per_cent = dict(kidneys=rows_real.datum_kidneys.median())
per_cent["liver"] = float(np.median([F.real_measure(b.geometry)["liver"] / b.geometry.rc["liver"] * b.geometry.liver_ml for b in bases[:20]]))
arithmetic = pd.DataFrame({k: dict(percent_ia_1h=per_cent[k], teff_h=targets.loc[f"teff_{k}", "value"], tiac_h=per_cent[k] / 100 * targets.loc[f"teff_{k}", "value"] / np.log(2), dose=C.self_dose(per_cent[k] / 100 * targets.loc[f"teff_{k}", "value"] / np.log(2), k, masses[k]), target=targets.loc[f"dose_{k}", "value"]) for k in ("kidneys", "liver")}).T
arithmetic["over_target"] = arithmetic.dose / arithmetic.target
arithmetic.to_csv(OUT / "nb19_dose_arithmetic.csv")
display(arithmetic.round(3))


def amount_job(job):
    i, nmol = job
    return C.forward(bases[i], tuple(theta_a), True, nmol)


def amount_build():
    with mp.get_context("fork").Pool(N_PROCESSES) as workers:
        return {nmol: C.population(workers.map(amount_job, [(i, nmol) for i in range(len(bases))], chunksize=4)) for nmol in (2.0, 100.0)}


amounts = cached("amount_check", amount_build)
saturation = pd.DataFrame(amounts).assign(ratio_100_over_2=lambda d: d[100.0] / d[2.0], target=targets.value)
saturation.to_csv(OUT / "nb19_amount_check.csv")
display(saturation[saturation.index.str.startswith("dose")].round(3))

## 8. Report of the first calibration (no agent factor). Decision taken after it: §9

**Sources.** Delker, Violet and Rosar give no organ or lesion half-life in what could be read (§1). Half-lives come from Sarnelli 2019 (mCRPC, planar, n = 9, with kidney and gland protection) and Peters 2022-2023 (one cohort, low-volume mHSPC, n = 10). There is **no spleen half-life, and lesion half-lives come from a single cohort with small lesions.** Tolerances from 2-3 studies are weak, so half-life tolerances carry an assumed floor of 0.10.

**Identifiability (§4), on simulation, by the rule set beforehand.**

| Parameter | Prior SD (log) | Posterior SD, half-lives / both | Recovery error, both | Rule, both |
|---|---|---|---|---|
| $\lambda_{rel}$ salivary | 0.86 | 0.34 / 0.24 | 0.28 | **passes** |
| $\lambda_{rel}$ kidneys | 0.86 | 0.43 / 0.31 | 0.35 | fails by a little |
| $\lambda_{rel}$ liver | 0.86 | 0.38 / 0.30 | 0.31 | fails by a little |
| $\lambda_{rel}$ lesions | 0.86 | 0.42 / 0.32 | 0.33 | fails by a little |
| $K_D$ | 2.31 | 2.06 / 1.37 | 1.41 | **no** |
| $\lambda_{int}$ | 1.65 | 1.52 / 0.81 | 0.93 | **no** |
| Lesion median $[R]$ | 1.65 | 1.63 / 1.39 | 1.22 | **no** |

Coverage of the 90% intervals is 0.88-0.94 with both target sets, so the posteriors are honest about this.

- **$K_D$ and $\lambda_{int}$ cannot be separated.** Their posterior correlation is +0.94, and doses respond to them with opposite sign and nearly equal size (-0.45, +0.50), so doses see about $\lambda_{int}/K_D$, the share of bound ligand that gets internalised. Half-lives respond to neither at the base (at most 0.14).
- **$\lambda_{rel}$ is pinned by half-lives only given the ligand.** At the base each half-life responds to its own $\lambda_{rel}$ alone (-0.67 to -0.82), so with the ligand fixed a tolerance of 0.10-0.18 gives about 0.13-0.23 in $\log\lambda_{rel}$ (computed from these slopes, not recovered by simulation). With the ligand free the half-life profile is open towards fast release (§5 figure), since a slow internalisation can then set the half-life instead. Doses alone identify none of the seven.
- **Lesion median:** one lesion dose with tolerance 0.81 and a slope of 0.36 cannot fix it.

**Calibration (§5), direct simulation, model over target.**

| | Kidneys | Liver | Spleen | Glands | Lesions | $\chi^2$ half-lives / doses |
|---|---|---|---|---|---|---|
| $T_{eff}$, base | 0.91 | 1.16 | | 0.76 | 0.68 | 31.2 |
| $T_{eff}$, arm A (whole body 0.92) | 1.21 | 1.02 | | 1.01 | 1.01 | **1.7** |
| Dose, base | 4.6 | 0.48 | 1.41 | 2.9 | 0.55 | 42.0 |
| **Dose, arm A: the agent mismatch** | **5.7** | **0.44** | **1.28** | **3.6** | **0.77** | **53.8** |
| Dose, arm B (lesion median 277) | 5.6 | 0.44 | 1.26 | 3.5 | 1.05 | 53.5 |
| Dose, arm C (all free) | 3.0 | 0.43 | 1.26 | 2.2 | 1.76 | 8.6 / 30.6 |

**Arm A values:** $\lambda_{rel}$ kidneys 1.94e-4, liver and spleen 3.38e-4, salivary 2.68e-4, lesions 8.7e-5 /min, against Kletting's I&T 2.84e-4, 2.84e-4, 3.86e-4, 1.66e-4: **slower for 617 on kidneys, glands and lesions, as published half-lives require.** The half-lives are met. The whole-body half-life, 36.8 h, is a little under 39.9 h. Arm A's kidney half-life at 1.21 is what closes it.

**Upper limits of the between-patient log SD of $\lambda_{rel}$ (§6):** kidneys 0.18-0.66, liver 0.34-0.50, salivary 0.15-0.28, lesions 0.35-0.46. These include measurement error. Drawing the lower values reproduces the published half-life spreads (0.13, 0.28, 0.11, 0.18).

**The ligand cannot absorb the residual (arm C).** Freeing everything drives $K_D$ to the edge of the box (28 of 30 nM) and the lesion median to its edge (3000), costs half-lives ($\chi^2$ 1.7 to 8.6, with kidneys 0.87, liver 1.30, whole body 0.80), and still leaves kidneys ×3.0, glands ×2.2, liver ×0.43. **The reason is structural: one ligand scales all receptor organs in the same direction, while the residual has opposite signs.** So these arm C values are not estimates, and they are not adopted.

**The dose residual as a measure of the agent mismatch (§7).**

- **Kidneys ×4-6.** This does not depend on the model: 14.3 %IA at 1 h on PSMA-11, times the published half-life, through the same S-value, gives ×4.4. Peters 2022 measured the same sign in patients, 2.21 ± 0.46 for the kidney dose predicted from PSMA-11 PET over SPECT, and scaled it empirically. Part of our excess is therefore a real PSMA-11 against 617 difference at 1 h, in the source they suggest; the rest is not explained. It also assumes retention of what the kidney holds at 1 h, so it is an upper limit.
- **Glands ×3.6:** the tolerance is wide (0.54), so this is 2.4 tolerances; the same reading applies, and the gland mask volume adds uncertainty.
- **Liver ×0.44: probably saturation.** At 2 nmol the same model gives 0.106 against a target of 0.115; at 100 nmol, 0.051. The fitted liver density is low (about 1.3 nmol/L), so the therapy amount saturates it, and spleen likewise. The real liver may hold non-saturable, non-receptor activity, which the model does not have. Hypothesis, not tested.
- **Lesions:** with the starting population ×0.55-0.77, and this is within tolerance (0.81). **The 30-50× of nb18 does not appear here,** because the real density population is no longer used. Lesion over kidney is still low because of the kidney excess.

### Proposal for the cohort, not applied

1. **$\lambda_{rel}$ from arm A, conditional on the ligand of Kletting,** with the SDs above and a declared dependence on that ligand.
2. **$K_D$ and $\lambda_{int}$ stay at the base.** Not estimable, and doses see only their ratio. Scenario grid rather than an estimate (nb16 §5).
3. **Lesion median:** the starting 136 is compatible with the lesion dose within tolerance; arm B (277) is a conditional value only. Not estimated.
4. **The kidney and gland dose excess is the open decision.** Three options: report the cohort with the excess, as a stated limitation; a declared per-organ agent factor on uptake at therapy (kidneys about 1/2.2 from Peters 2022, leaving the rest as residual); or a cohort labelled for PSMA-11-like kinetics only. **The choice sets the absolute kidney dose of the virtual cohort.** It does not need more fitting.

### Limitations

- Self-dose only; no crossfire, which would matter most for liver and spleen. Monte Carlo later.
- Population level: median half-lives and mean doses over 66 patients, lesions at or above 0.5 mL. Satellites at therapy; 100 nmol assumed. Whole-body half-life counts released ligand as excreted.
- Published doses mix planar and SPECT and overlap (Ells 2024 pools earlier studies), so tolerances are if anything small.
- Patients are the point estimates of nb18, including the kidney density that did not converge (51 to 44) and is likely 11-15% low on simulation. The lesion likelihood carries the mask term.
- The emulator error is under half a tolerance on all outputs (worst, lesion half-life, 0.45). Arm results come from direct simulation.
- Recovery truths are uniform in the focused box; identifiability near the real targets may differ. The "separable" rule is a threshold of ours.

## 9. Agent factor per organ (option B), calibrated

**Decision after §8.** A factor $\phi$ per organ, therapy PSMA-617 uptake over PET PSMA-11 uptake, for kidneys, glands, liver and spleen, estimated in the population calibration together with $\lambda_{rel}$. Half-lives keep priority. Published doses set the factors: Ells 2024, Rosar 2021, Ells 2026. Peters 2022 (2.21 on the kidney) is a check of sign and order of magnitude, **not a target**. Between-patient spread of $\phi$: a sensitivity setting, log SD 0 and 0.3, as no data fixes it. $\phi=1$ stays as the control arm (options A and C of §8).

**This agent difference is information the 1 h PET cannot carry.** The PET contains PSMA-11 only: it measures uptake of that agent, and no fit to it can say how another agent would behave in the same patient. So $\phi$ is set by published doses from other patients at population level, and it does not depend on this patient's PET. The per-patient information on $\phi$ is none, and any per-patient spread of it is assumed.

**How it acts.** $\phi$ multiplies the organ density fitted under the PET ligand, at therapy only, so mass balance, plasma and saturation follow. It is an effective capacity for the therapy agent. Uptake is nonlinear in density, so the **uptake factor** it implies (dose at $\phi$ over dose at 1, same $\lambda_{rel}$) is reported as well, and this is what compares with Peters. $K_D$ and $\lambda_{int}$ at the base (§4: not separable).

**Targets.** Doses: geometric mean of Ells 2024, Rosar 2021, Ells 2026, with the tolerance their between-study log SD. **Those three give no spleen dose**: the spleen target is the one exception, Violet 2019 and Delker 2016. Half-lives as in §1 on organs and whole body.

**Lesion $\lambda_{rel}$: wide prior.** The lesion half-life comes from **one cohort of low-volume hormone-sensitive patients with small lesions, not from mCRPC**, so it is not used as a target with tolerance 0.10. Prior on $\log\lambda_{rel}$ centred on the value from that half-life, 8.7e-5 /min, log SD 0.7 (assumption; about 28-122 h in half-life over ±2 SD, and Kletting's I&T 1.66e-4 inside 1 SD). **Lesion median $[R]$:** start population with its five-pair error, $\log$ 136 ± 0.62, and the lesion dose as the anchor.

In [ ]:
DOSE_SOURCES = ("Ells2024", "Rosar2021", "Ells2026")
rows = []
for key in C.OUTPUTS:
    if key.startswith("teff"):
        rows.append(dict(target=key, value=targets.loc[key, "value"], tolerance=targets.loc[key, "tolerance"], used=key != "teff_lesions"))
    else:
        values = [v for k, v in DOSES[key].items() if k in DOSE_SOURCES] if key != "dose_spleen" else list(DOSES[key].values())
        rows.append(dict(target=key, value=float(np.exp(np.mean(np.log(values)))), tolerance=float(np.log(values).std(ddof=1)), used=True))
agent_targets = pd.DataFrame(rows).set_index("target")
agent_targets.to_csv(OUT / "nb19_agent_targets.csv")
display(agent_targets.round(3))
YA, TA = np.log(agent_targets.value.to_numpy()), agent_targets.tolerance.to_numpy()
USED = agent_targets.used.to_numpy()
LOW_A, HIGH_A = C.AGENT_BOX[:, 0], C.AGENT_BOX[:, 1]
PRIOR_LESION = (np.log(8.7e-5), 0.7)
PRIOR_MEDIAN = (np.log(F.TUMOUR_MEDIAN), float(F.TUMOUR_SD / np.sqrt(F.FIVE_PATIENTS)))
print("Median prior SD", round(PRIOR_MEDIAN[1], 2))


def one_agent(job):
    index, theta, spread = job
    return C.forward_agent(bases[index], theta, spread)


def evaluate_agent(thetas, workers, spread=0.0):
    thetas = np.atleast_2d(thetas)
    output = workers.map(one_agent, [(i, tuple(t), spread) for t in thetas for i in range(len(bases))], chunksize=len(bases))
    out = np.zeros((len(thetas), len(C.OUTPUTS)))
    for k in range(len(thetas)):
        pop = C.population(output[k * len(bases): (k + 1) * len(bases)])
        out[k] = [np.log(pop[n]) for n in C.OUTPUTS]
    return out


def agent_design_build():
    rng = np.random.default_rng(SEED + 5)
    x = C.latin_hypercube(1500, rng, C.AGENT_BOX)
    hold = C.latin_hypercube(120, rng, C.AGENT_BOX)
    hold[:, 7] = np.clip(rng.normal(*PRIOR_LESION, 120), LOW_A[7], HIGH_A[7])
    hold[:, 8] = np.clip(rng.normal(*PRIOR_MEDIAN, 120), LOW_A[8], HIGH_A[8])
    with mp.get_context("fork").Pool(N_PROCESSES) as workers:
        return dict(x=x, y=evaluate_agent(x, workers), hold_x=hold, hold_y=evaluate_agent(hold, workers))


agent_design = cached("agent_design", agent_design_build)
unit_a = lambda x: (x - LOW_A) / (HIGH_A - LOW_A)
emulator_a = RBFInterpolator(unit_a(agent_design["x"]), agent_design["y"], kernel="thin_plate_spline", smoothing=1e-4)
holdout_a = emulator_a(unit_a(agent_design["hold_x"])) - agent_design["hold_y"]
accuracy_a = pd.DataFrame(dict(rms=np.sqrt((holdout_a**2).mean(axis=0)), p95=np.quantile(np.abs(holdout_a), 0.95, axis=0), tolerance=TA), index=C.OUTPUTS)
accuracy_a["rms_over_tolerance"] = accuracy_a.rms / accuracy_a.tolerance
accuracy_a.to_csv(OUT / "nb19_agent_emulator_accuracy.csv")
display(accuracy_a.round(3))


def log_posterior_a(points, observed):
    inside = np.all((points >= LOW_A) & (points <= HIGH_A), axis=1)
    out = np.full(len(points), -np.inf)
    if inside.any():
        p = points[inside]
        z = (emulator_a(unit_a(p)) - observed)[:, USED] / TA[USED]
        out[inside] = -0.5 * np.sum(z**2, axis=1) - 0.5 * ((p[:, 7] - PRIOR_LESION[0]) / PRIOR_LESION[1]) ** 2 - 0.5 * ((p[:, 8] - PRIOR_MEDIAN[0]) / PRIOR_MEDIAN[1]) ** 2
    return out


def sample_a(observed, seed, n_steps=900, walkers=64):
    rng = np.random.default_rng(seed)
    start = C.latin_hypercube(walkers * 40, rng, C.AGENT_BOX)
    lp = log_posterior_a(start, observed)
    start = start[np.argsort(lp)[-walkers:]] + rng.normal(0, 0.05, (walkers, len(LOW_A)))
    start = np.clip(start, LOW_A + 1e-3, HIGH_A - 1e-3)
    chain, acceptance = C.stretch_sampler(lambda p: log_posterior_a(p, observed), start, n_steps, rng)
    return chain[n_steps // 2 :].reshape(-1, len(LOW_A))[::5], acceptance

## 10. Identifiability with the factors, before the fit

Same check as §4. The 120 held-out populations are truths, lesion $\lambda_{rel}$ and median from their priors, observed with noise at the tolerances. Rule of §4 against the box SD.

In [ ]:
def recovery_job_a(job):
    index, seed = job
    rng = np.random.default_rng(seed)
    observed = agent_design["hold_y"][index] + rng.normal(0, TA)
    draws, acceptance = sample_a(observed, seed)
    truth = agent_design["hold_x"][index]
    q = np.quantile(draws, [0.05, 0.5, 0.95], axis=0)
    return dict(truth=truth, median=q[1], sd=draws.std(axis=0), inside=(q[0] <= truth) & (truth <= q[2]))


def recovery_build_a():
    with mp.get_context("fork").Pool(N_PROCESSES) as workers:
        return workers.map(recovery_job_a, [(i, SEED + 3000 + i) for i in range(120)], chunksize=2)


recovery_a = cached("agent_recovery", recovery_build_a)
box_sd = (HIGH_A - LOW_A) / np.sqrt(12)
error = np.array([r["median"] - r["truth"] for r in recovery_a])
sd_a = np.array([r["sd"] for r in recovery_a])
inside_a = np.array([r["inside"] for r in recovery_a])
prior_sd = box_sd.copy()
prior_sd[7], prior_sd[8] = PRIOR_LESION[1], PRIOR_MEDIAN[1]
identifiability_a = pd.DataFrame(dict(box_sd=box_sd, prior_sd=prior_sd, rms_error=np.sqrt((error**2).mean(axis=0)), posterior_sd=np.median(sd_a, axis=0), coverage_90=inside_a.mean(axis=0)), index=C.AGENT_NAMES)
identifiability_a["separable"] = (identifiability_a.rms_error < 0.5 * identifiability_a.prior_sd) & (identifiability_a.posterior_sd < identifiability_a.prior_sd / 3)
identifiability_a.to_csv(OUT / "nb19_agent_identifiability.csv")
display(identifiability_a.round(3))

## 11. The calibrated population

Sampler on the emulator for the intervals. The maximum is then polished by least squares on the **direct** simulation, for log SD of $\phi$ 0 and 0.3 between patients. Control: $\phi=1$, the rest refitted, same priors and targets. All tables below are direct simulation.

In [ ]:
chain_a, acceptance_a = cached("agent_posterior", lambda: sample_a(YA, SEED + 7, n_steps=3000, walkers=128))
print("Acceptance", round(acceptance_a, 2))
map_a = chain_a[np.argmax(log_posterior_a(chain_a, YA))]


def polish(start, spread, free, workers):
    def residual(x):
        theta = start.copy()
        theta[free] = x
        y = evaluate_agent(theta, workers, spread)[0]
        extra = [(theta[7] - PRIOR_LESION[0]) / PRIOR_LESION[1], (theta[8] - PRIOR_MEDIAN[0]) / PRIOR_MEDIAN[1]]
        return np.r_[((y - YA) / TA)[USED], extra]
    from scipy.optimize import least_squares
    solution = least_squares(residual, start[free], bounds=(LOW_A[free], HIGH_A[free]), diff_step=0.02, xtol=1e-3, ftol=1e-3, max_nfev=25)
    theta = start.copy()
    theta[free] = solution.x
    return theta, solution


def arms_build():
    out = {}
    all_free = np.arange(9)
    no_factor = np.arange(4, 9)
    control = map_a.copy()
    control[:4] = 0.0
    with mp.get_context("fork").Pool(N_PROCESSES) as workers:
        out["agent, spread 0"], s0 = polish(map_a, 0.0, all_free, workers)
        out["agent, spread 0.3"], s3 = polish(out["agent, spread 0"], 0.3, all_free, workers)
        out["control, factor 1"], _ = polish(control, 0.0, no_factor, workers)
        direct = {k: evaluate_agent(v, workers, 0.3 if k.endswith("0.3") else 0.0)[0] for k, v in out.items()}
        without = out["agent, spread 0"].copy()
        without[:4] = 0.0
        uptake = evaluate_agent(without, workers)[0]
    return dict(theta=out, direct=direct, uptake_reference=uptake, jacobian=s0.jac)


arms_a = cached("agent_arms", arms_build)
q = np.quantile(chain_a, [0.05, 0.5, 0.95], axis=0)
population = pd.DataFrame(dict(prior_or_box=[f"box {np.exp(LOW_A[i]):.3g}-{np.exp(HIGH_A[i]):.3g}" for i in range(7)] + [f"8.7e-05, log SD {PRIOR_LESION[1]}", f"136, log SD {PRIOR_MEDIAN[1]:.2f}"], agent_spread_0=np.exp(arms_a["theta"]["agent, spread 0"]), q05=np.exp(q[0]), q50=np.exp(q[1]), q95=np.exp(q[2]), agent_spread_03=np.exp(arms_a["theta"]["agent, spread 0.3"]), control_factor_1=np.exp(arms_a["theta"]["control, factor 1"])), index=C.AGENT_NAMES)
population.to_csv(OUT / "nb19_calibrated_population.csv")
display(population.map(lambda v: float(f"{v:.3g}") if not isinstance(v, str) else v))
ratios = pd.DataFrame({k: np.exp(v - YA) for k, v in arms_a["direct"].items()}, index=C.OUTPUTS).assign(target=agent_targets.value, tolerance=TA, used=USED)
ratios.to_csv(OUT / "nb19_calibrated_residuals.csv")
display(ratios.round(3))
display(pd.DataFrame({k: dict(half_lives=float(np.sum((((v - YA) / TA)[USED & T_ONLY]) ** 2)), doses=float(np.sum((((v - YA) / TA)[~T_ONLY]) ** 2))) for k, v in arms_a["direct"].items()}).round(2))
dose_index = [C.OUTPUTS.index(f"dose_{k}") for k in C.AGENT_ORGANS]
uptake_factor = pd.DataFrame(dict(density_factor=np.exp(arms_a["theta"]["agent, spread 0"][:4]), q05=np.exp(q[0][:4]), q95=np.exp(q[2][:4]), uptake_factor_therapy_over_pet=np.exp(arms_a["direct"]["agent, spread 0"][dose_index] - arms_a["uptake_reference"][dose_index])), index=C.AGENT_ORGANS)
uptake_factor["pet_over_therapy"] = 1 / uptake_factor.uptake_factor_therapy_over_pet
uptake_factor.to_csv(OUT / "nb19_uptake_factor.csv")
display(uptake_factor.round(3))
print("Check, not a target. Peters 2022, kidney dose predicted from PSMA-11 PET over SPECT: 2.21 +- 0.46. Here:", round(uptake_factor.loc["kidneys", "pet_over_therapy"], 2))
display(pd.DataFrame(np.corrcoef(chain_a.T), index=C.AGENT_NAMES, columns=C.AGENT_NAMES).round(2))


def spread_rows(job):
    i, spread = job
    return C.forward_agent(bases[i], arms_a["theta"][f"agent, spread {spread:g}"], spread)


def per_patient_build():
    with mp.get_context("fork").Pool(N_PROCESSES) as workers:
        return {s: workers.map(spread_rows, [(i, s) for i in range(len(bases))], chunksize=4) for s in (0.0, 0.3)}


per_patient = cached("agent_per_patient", per_patient_build)
display(pd.DataFrame({f"log SD of dose between patients, spread {s:g}": {k: np.log([r[k] for r in rows_ if k in r]).std() for k in C.OUTPUTS if k.startswith("dose")} for s, rows_ in per_patient.items()}).round(3))

## 12. Report: the calibrated population. No cohort generated; stopped here

All stages direct simulation on 66 patients of the training split. Priorities as decided: half-lives first, the factors set by Ells 2024, Rosar 2021, Ells 2026, Peters 2022 as a check.

| Parameter | Value (spread 0) | 90% interval | Spread 0.3 | Control, factor 1 | Identified? |
|---|---|---|---|---|---|
| $\phi$ kidneys, on density | **0.20** | 0.09-0.37 | 0.20 | 1 | yes |
| $\phi$ glands | **0.25** | 0.09-1.10 | 0.24 | 1 | **no**, tolerance 0.67, interval includes 1 |
| $\phi$ liver | **2.3** | 1.5-3.7 | 2.0 | 1 | yes |
| $\phi$ spleen | **0.95** | 0.46-1.8 | 0.94 | 1 | yes, compatible with 1; target only Violet, Delker |
| $\lambda_{rel}$ kidneys (/min) | **1.95e-4** | 1.24-2.62e-4 | 1.89e-4 | 2.36e-4 | yes |
| $\lambda_{rel}$ liver, spleen | **3.06e-4** | 2.32-4.06e-4 | 3.09e-4 | 2.79e-4 | yes |
| $\lambda_{rel}$ glands | **2.69e-4** | 2.15-3.31e-4 | 2.69e-4 | 2.78e-4 | yes |
| $\lambda_{rel}$ lesions | **6.4e-5** | 2.7-15e-5 | 6.7e-5 | 6.6e-5 | **no, prior** (log SD 0.56 from 0.7) |
| Lesion median $[R]$ (nmol/L) | **169** | 78-457 | 141 | 188 | **no, prior** (log SD 0.54 from 0.62) |

Identifiability: 120 truths with noise at the tolerances (§10), coverage of the 90% interval 0.88-0.93, except lesion $\lambda_{rel}$ at 0.82. The posterior correlations are small (at most 0.31, $\phi$ kidneys with $\lambda_{rel}$ kidneys), so each factor is set by its own dose. $K_D$ and $\lambda_{int}$ remain at the base, not estimated.

**Fit, direct simulation, model over target.**

| | Kidneys | Liver | Spleen | Glands | Lesions | Whole body | $\chi^2$ half-lives / doses |
|---|---|---|---|---|---|---|---|
| Half-life, agent | 1.20 | 1.10 | | 1.00 | (1.17, not a target) | 0.83 | **4.7** |
| Half-life, control | 1.05 | 1.18 | | 0.98 | (1.16) | 0.85 | **4.0** |
| Dose, agent | 1.15 | 0.95 | 0.99 | 0.98 | 0.80 | | **0.45** |
| Dose, control | 4.9 | 0.46 | 1.10 | 3.5 | 0.80 | | **30.9** |

**How to read it.**

- **The good dose fit is not a validation.** Four factors for four doses fit by construction. What the calibration can tell is the size of the factors and that **half-lives barely move** ($\chi^2$ 4.0 to 4.7). Whole body stays low, 33 h against 39.9, in both arms.
- **Each $\phi$ absorbs everything between this PET fit and those doses, not only the agent:** the kidney datum and non-convergent $[R_K]$, self-dose without crossfire, the assumed 100 nmol, and the method spread of the sources. So $\phi$ is an effective factor. Its name, agent factor, states the leading reading, not a demonstration.
- **Kidneys against Peters, check only.** Implied PET over therapy uptake 4.8, Peters 2022 2.21 ± 0.46: **same sign, same order, about twice.** The Peters value corresponds to an uptake factor of 0.45, just above the upper end of this interval. Peters: low-volume mHSPC, one centre, and PET and SPECT from the same patients. No other organ has such a check; Peters found no such constant factor on the other organs.
- **Liver above 1** is the opposite direction. This fits the hypothesis of §7, that 100 nmol saturate a fitted low density while real liver may hold non-receptor activity. A factor on capacity reproduces the mean dose, but not necessarily the dependence on the amount.
- **Glands:** the factor is effectively a statement of the between-study spread. Ells 2026 is about a third of Ells 2024 and Rosar.
- **Lesions:** $\lambda_{rel}$ and median are essentially priors, from one hormone-sensitive low-volume cohort for $\lambda_{rel}$ and five lesion pairs on I&T for the median. The lesion dose is at 0.80 of target, inside its tolerance. **From the real PET, lesions keep only flow and the lower bound.**

**Spread of $\phi$ between patients, as a setting (log SD 0 and 0.3).** Medians barely change; liver 2.3 to 2.0, lesion median 169 to 141. Log SD of dose between patients, spread 0 to 0.3: kidneys 0.85 to 0.92, liver 0.30 to 0.43, spleen 0.44 to 0.46, glands 0.16 to 0.34, lesions 1.16 to 1.21. As a rough comparison, not a target, published relative SDs (Rosar, Ells 2026) are about 0.4-0.5 for kidneys, 0.5-0.8 for liver and 0.4-0.8 for glands. So glands and liver are closer at 0.3, while **kidneys are over-dispersed in both settings**, inherited from the $[R_K]$ spread of 1.1 from nb18.

**This agent difference is information the 1 h PET cannot carry.** It enters as a population constant, the same for a patient whatever their PET shows.

**Between-patient log SD of $\lambda_{rel}$:** §6, upper limits, not recalibrated with the factors.

### Limitations added to §8

1. Factors fixed by 3 sources (Violet and Delker for the spleen), which disagree, with an overlap: Ells 2024 is a meta-analysis.
2. One factor per organ, on density. A factor on uptake independent of amount would behave differently away from 100 nmol.
3. Half-life and dose targets come from different patients and different agents' protocols; Sarnelli gave mannitol and glutamate, which lower kidney and gland uptake.
4. Factors act at population level, through point estimates per patient, not through the posterior of nb18. The cohort will draw from that posterior; **a check that the means hold when drawing is still to do.**

## 13. Before the cohort: what $\phi$ is, and the whole-body half-life

**$\phi$ is an effective factor.** It absorbs everything between this PET fit and the published doses: the non-convergence of $[R_K]$, the missing crossfire, the assumed 100 nmol, the mixed methods of the sources, and the difference between agents. **The kidney $\phi$ implies a PET over therapy uptake ratio of about 5, against 2.2 in Peters 2022: same sign, twice the size.**

**Whole-body $T_{eff}$ is 15-17% short in both arms.** Checked below against urinary excretion and GFR. Kurth 2018 (mCRPC, n = 50, external probe), Table 2, cumulative excreted share of administered activity: 30% at 2 h, 50% at 4 h, 70% at 12 h, 76% at 24 h, 77-78% from 48 h. Text: about 4.8 of 6.6 GBq over 72 h. GFR raised by 30% and tumour burden are the two physical candidates tested.

In [ ]:
import pbpk_cohort as Co
N_PROCESSES = 3
theta_agent = arms_a["theta"]["agent, spread 0"]


def wb_job(job):
    i, scale = job
    return Co.whole_body_check(bases[i], theta_agent, scale)


def wb_build():
    with mp.get_context("fork").Pool(N_PROCESSES) as workers:
        return {scale: workers.map(wb_job, [(i, scale) for i in range(len(bases))], chunksize=4) for scale in (1.0, 1.3)}


wb = {k: pd.DataFrame(v) for k, v in cached("whole_body", wb_build).items()}
kurth = {2: 0.30, 4: 0.50, 12: 0.70, 24: 0.76, 48: 0.77, 72: 0.78}
excretion = pd.DataFrame({"Kurth 2018": kurth, "model": {h: wb[1.0][f"excreted_{h}h"].median() for h in kurth}, "model, GFR x 1.3": {h: wb[1.3][f"excreted_{h}h"].median() for h in kurth}})
excretion.to_csv(OUT / "nb19_excretion.csv")
display(excretion.round(3))
d = wb[1.0]
burden = d.groupby(pd.cut(d.tlv_ml, [-1, 10, 100, 300, 1e5]), observed=True).agg(n=("tlv_ml", "size"), teff_whole_body=("teff_whole_body", "median"), excreted_24h=("excreted_24h", "median"), excreted_72h=("excreted_72h", "median"))
burden.to_csv(OUT / "nb19_whole_body_by_burden.csv")
display(burden.round(3))
print("Whole-body T_eff, h: target", round(agent_targets.loc["teff_whole_body", "value"], 1), "| model median", round(d.teff_whole_body.median(), 1), "| GFR x 1.3", round(wb[1.3].teff_whole_body.median(), 1))

**Reading: no physically motivated fix; declared.** Three facts.
- **Tumour burden accounts for most of the shortfall.** The whole-body half-life grows with TLV in the model, 31.3 h below 10 mL and 37.8 h above 300 mL, against the target 39.9. The published cohorts are mCRPC; our PSMA-11 training cohort has a median TLV near 10 mL, so a shorter half-life in the median patient is expected.
- **GFR is not the cause.** Raising it by 30% matches the early excretion (47.6% against 50% at 4 h) and lengthens the half-life to 35.8 h, but pushes late excretion further from the data (89% against 76% at 24 h). GFR stays as printed in Table A, with the Fresco limitation.
- **The model excretes more than Kurth at late times, at every burden** (24 h: 83-86% against 76%; 72 h: 88-91% against 78%). The real body retains about twice as much at 72 h as the model. Not resolved: part may lie in Kurth's method (external probe, excreted activity computed from dose rate), and part in retention the model lacks. The agent factor on kidneys lowers retention further.

So the whole-body half-life and late retention stay as declared limitations. The cohort's dose does not rest on them, since organ doses are calibrated directly.

## 14. Between-patient spread as a target

**Why.** With point estimates the kidney dose spread is log SD 0.85-0.92, against about 0.35 in the literature: the population SD of $[R_K]$ of nb18 (1.1) mixes biology and an undeclared error of the kidney datum.

**Targets.** Published relative SD of Gy/GBq per patient, as a log SD $\sqrt{\ln(1+CV^2)}$, averaged over sources. Read: Rosar 2021 (3D), Ells 2026, Delker 2016 (abstract), Peters 2022 (same cohort as Peters 2023). **Ells 2024 gives no patient-level SD (pooled means and CI between studies); Violet 2019 gives it in Table 1, whose cells could not be read** (the journal refused the table page). These SDs include measurement error, so they are upper limits of the biological spread; our spread is compared with them as published.

**What they constrain.** Kidneys: the population SD $\tau$ of $[R_K]$ on a grid. At each $\tau$ the median and an **explicit error of the kidney datum** are refitted by marginal likelihood on the grids of the final pass of nb18, so that what is no longer biology is declared as datum error, not dropped. Liver, spleen, glands: the spread of $\phi$ on a grid. Glands keep 0.3 if the target confirms it. **Draws come from the posterior of each phantom, as the cohort will.** $\lambda_{rel}$ spreads: the lower values of §6.

In [ ]:
log_sd = lambda cv: float(np.sqrt(np.log(1 + cv**2)))
SPREAD_SOURCES = {
    "dose_kidneys": dict(Ells2026=0.11 / 0.29, Rosar2021=0.28 / 0.54, Delker2016=0.6 / 2.2, Peters=0.11 / 0.49),
    "dose_liver": dict(Ells2026=0.09 / 0.11, Rosar2021=0.05 / 0.10, Delker2016=0.2 / 0.4, Peters=0.01 / 0.09),
    "dose_spleen": dict(Delker2016=0.1 / 0.4),
    "dose_salivary": dict(Ells2026=np.mean([0.15 / 0.26, 0.18 / 0.23]), Rosar2021=np.mean([0.34 / 0.81, 0.39 / 0.72]), Delker2016=1.8 / 5.1, Peters=0.17 / 0.39),
    "dose_lesions": dict(Ells2026=np.mean([1.80 / 1.74, 3.40 / 3.21]), Rosar2021=1.32 / 1.68),
}
spread_targets = pd.DataFrame({k: {**{s: log_sd(c) for s, c in v.items()}, "target": float(np.mean([log_sd(c) for c in v.values()]))} for k, v in SPREAD_SOURCES.items()}).T
spread_targets.to_csv(OUT / "nb19_spread_targets.csv")
display(spread_targets.round(3))
SPREAD = spread_targets.target.to_dict()
final = pickle.loads((OUT / "nb18_eb_real.pkl").read_bytes())
results_final = final["fits"][-1]["likelihood_results"]
history = final["history"][-1]
first_study = {}
for r in results_final:
    first_study.setdefault(r["subject"], r)
chosen_ids = [r["study_id"] for r in list(first_study.values())[::2]]
TAUS = [0.2, 0.3, 0.44, 0.7]
kidney_grid = pd.DataFrame(cached("kidney_grid", lambda: [Co.kidney_population(results_final, t) for t in TAUS]))
kidney_grid.to_csv(OUT / "nb19_kidney_population_grid.csv", index=False)
display(kidney_grid.round(3))
del results_final, first_study, final
HYPER = replace_hyper = F.LesionHyper(mu=float(theta_agent[8]))
MASK = F.LesionOffset("mask", (history["mask_b"],))


def settings_for(row):
    return dict(priors={**history["priors"], "kidneys": (row["median"], row["tau"])}, offsets=history["offsets"], blood_offsets=history["blood"], lesion_offset=MASK, extra_sigma=dict(liver=F.ORGAN_IMAGE_SD["liver"], kidneys=row["datum_sigma"]))


def run_bundles(ids, settings, hyper, keep):
    with mp.get_context("fork").Pool(N_PROCESSES) as workers:
        return workers.map(Co.stage_bundle, [(s, settings, hyper, keep) for s in ids], chunksize=2)


REL_SD = [float(spreads.loc[k, "log_sd_lambda_low"]) for k in ("kidneys", "liver", "salivary", "lesions")]
print("lambda_rel log SD:", np.round(REL_SD, 3))
NONE = [0.0, 0.0, 0.0, 0.0]


def spread_eval_job(job):
    index, theta, phi_sd, rel_sd = job
    return Co.forward_spread(CURRENT[index], theta, phi_sd, rel_sd)


def spread_eval(base_list, theta, phi_sd, rel_sd):
    global CURRENT
    CURRENT = base_list
    with mp.get_context("fork").Pool(N_PROCESSES) as workers:
        return workers.map(spread_eval_job, [(i, tuple(theta), tuple(phi_sd), tuple(rel_sd)) for i in range(len(base_list))], chunksize=6)


def summary_of(rows):
    return {k: float(np.log([r[k] for r in rows if k in r]).std()) for k in SPREAD}


def tau_build():
    out = {}
    for _, row in kidney_grid.iterrows():
        bundles = run_bundles(chosen_ids, settings_for(row), HYPER, 3)
        priors = settings_for(row)["priors"]
        rows = spread_eval(Co.calibration_bases(bundles, priors), theta_agent, NONE, REL_SD)
        out[row["tau"]] = dict(spread=summary_of(rows), mean_dose_kidneys=float(np.mean([r["dose_kidneys"] for r in rows])), ess=float(np.median([b["ess"] for b in bundles])), z_kidneys=float(np.std([b["z"]["kidneys"] for b in bundles])), r_kidneys=float(np.median([np.exp(b["draws"][:, 0]).mean() for b in bundles])))
    return out


tau_results = cached("tau_results", tau_build)
tau_table = pd.DataFrame({t: {"dose_kidneys log SD": v["spread"]["dose_kidneys"], "mean dose kidneys": v["mean_dose_kidneys"], "z SD of the kidney datum": v["z_kidneys"], "importance ESS of 300": v["ess"]} for t, v in tau_results.items()}).T
tau_table.to_csv(OUT / "nb19_tau_results.csv")
display(tau_table.round(3))
TAU = min(tau_results, key=lambda t: abs(tau_results[t]["spread"]["dose_kidneys"] - SPREAD["dose_kidneys"]))
kidney_row = kidney_grid[kidney_grid.tau == TAU].iloc[0]
print("Chosen tau:", TAU, "| target", round(SPREAD["dose_kidneys"], 3))
FINAL_SETTINGS = settings_for(kidney_row)
bundles_final = cached("bundles_final", lambda: run_bundles(chosen_ids, FINAL_SETTINGS, HYPER, 3))
bases_final = Co.calibration_bases(bundles_final, FINAL_SETTINGS["priors"])
PHI_GRID = [0.0, 0.15, 0.3, 0.45]
phi_rows = cached("phi_grid", lambda: {s: summary_of(spread_eval(bases_final, theta_agent, [0.0, s, s, s], REL_SD)) for s in PHI_GRID})
phi_table = pd.DataFrame(phi_rows).T.assign(**{})
phi_table.to_csv(OUT / "nb19_phi_spread_grid.csv")
display(phi_table.round(3))
print("Targets:", {k: round(v, 3) for k, v in SPREAD.items()})
pick = lambda key: min(PHI_GRID, key=lambda s: abs(phi_rows[s][key] - SPREAD[key]))
PHI_SD = dict(kidneys=0.0, salivary=pick("dose_salivary"), liver=pick("dose_liver"), spleen=pick("dose_spleen"))
print("Chosen spread of phi:", PHI_SD)

## 15. Final population for the cohort

The medians recalibrated by least squares on direct simulation, with posterior draws and the spreads above, same targets and priors as §11. Control at factor 1 likewise, with $\phi$ spread 0. The lesion $\lambda_{rel}$ median stays uncertain: each virtual patient draws its own lesion median around the calibrated one with the posterior SD of §11, so the wide prior reaches the cohort. **Population saved for nb20.**

In [ ]:
from scipy.optimize import least_squares


def recalibrate(start, free, phi_sd, base_list):
    global CURRENT
    CURRENT = base_list
    with mp.get_context("fork").Pool(N_PROCESSES) as workers:
        def residual(x):
            theta = start.copy()
            theta[free] = x
            rows = workers.map(spread_eval_job, [(i, tuple(theta), tuple(phi_sd), tuple(REL_SD)) for i in range(len(base_list))], chunksize=6)
            pop = C.population(rows)
            y = np.array([np.log(pop[n]) for n in C.OUTPUTS])
            return np.r_[((y - YA) / TA)[USED], (theta[7] - PRIOR_LESION[0]) / PRIOR_LESION[1], (theta[8] - PRIOR_MEDIAN[0]) / PRIOR_MEDIAN[1]]
        solution = least_squares(residual, start[free], bounds=(LOW_A[free], HIGH_A[free]), diff_step=0.03, xtol=2e-3, ftol=2e-3, max_nfev=12)
    theta = start.copy()
    theta[free] = solution.x
    return theta


PHI_LIST = [PHI_SD[k] for k in C.AGENT_ORGANS]
theta_main = cached("theta_main", lambda: recalibrate(theta_agent.copy(), np.arange(9), PHI_LIST, bases_final))
control_start = arms_a["theta"]["control, factor 1"].copy()
theta_control = cached("theta_control", lambda: recalibrate(control_start, np.arange(4, 9), NONE, bases_final))
main_rows = cached("rows_main", lambda: spread_eval(bases_final, theta_main, PHI_LIST, REL_SD))
control_rows = cached("rows_control", lambda: spread_eval(bases_final, theta_control, NONE, REL_SD))
LESION_EPISTEMIC = float(chain_a[:, 7].std())
check_a = pd.DataFrame({"target": agent_targets.value, "tolerance": agent_targets.tolerance, "used": agent_targets.used, "main": pd.Series(C.population(main_rows)), "control": pd.Series(C.population(control_rows))})
check_a["main ratio"] = check_a.main / check_a.target
check_a["control ratio"] = check_a.control / check_a.target
check_a.to_csv(OUT / "nb19_final_against_targets.csv")
display(check_a.round(3))
check_s = pd.DataFrame({"target": pd.Series(SPREAD), "main": pd.Series(summary_of(main_rows)), "control": pd.Series(summary_of(control_rows))})
check_s.to_csv(OUT / "nb19_final_spread.csv")
display(check_s.round(3))
final_population = pd.DataFrame(dict(main=np.exp(theta_main), control=np.exp(theta_control), before_spreads=np.exp(theta_agent)), index=C.AGENT_NAMES)
final_population.to_csv(OUT / "nb19_final_population.csv")
display(final_population.map(lambda v: float(f"{v:.3g}")))
print("tau", TAU, "| kidneys median", round(kidney_row["median"], 2), "| datum sigma", round(kidney_row["datum_sigma"], 3), "| phi SD", PHI_SD, "| lambda_rel SD", np.round(REL_SD, 3), "| lesion epistemic SD", round(LESION_EPISTEMIC, 3))
(OUT / "nb19_cohort_population.pkl").write_bytes(pickle.dumps(dict(main=dict(theta=theta_main, phi_sd=PHI_SD), control=dict(theta=theta_control, phi_sd=dict(zip(C.AGENT_ORGANS, NONE))), rel_sd=dict(zip(("kidneys", "liver", "salivary", "lesions"), REL_SD)), lesion_epistemic_sd=LESION_EPISTEMIC, settings=FINAL_SETTINGS, hyper_median=float(theta_main[8]), hyper_median_control=float(theta_control[8]), tau=TAU, spread_targets=SPREAD, targets=agent_targets)))

In [ ]:
without_phi = theta_main.copy()
without_phi[:4] = 0.0
rows_without = cached("rows_without_phi", lambda: spread_eval(bases_final, without_phi, NONE, REL_SD))
implied = pd.DataFrame({k: dict(with_phi=np.mean([r[f"dose_{k}"] for r in main_rows if f"dose_{k}" in r]), phi_1_same_lambda=np.mean([r[f"dose_{k}"] for r in rows_without if f"dose_{k}" in r])) for k in C.AGENT_ORGANS}).T
implied["pet_over_therapy"] = implied.phi_1_same_lambda / implied.with_phi
implied.to_csv(OUT / "nb19_final_uptake_ratio.csv")
display(implied.round(3))
print("Check, not a target: Peters 2022, kidney 2.21 +- 0.46")

## 16. Report of §13-§15: the population given to the cohort

**$\phi$ is an effective factor.** It absorbs the non-convergence of $[R_K]$, the missing crossfire, the assumed 100 nmol, the mixed methods of the sources, as well as the difference between agents. **Kidney PET over therapy uptake: about 5 against 2.2 in Peters 2022 before the spread target (§12): same sign, twice the size. After the spread target, 3.1 (§15):** still the same sign, closer. The move is itself a measure of how much of the kidney factor was $[R_K]$ and not agent: with $\tau$ 0.3 the median density falls from 43.9 to 30.1.

**Kidney spread.** The spread target of log SD 0.34 selects $\tau=0.30$ on the grid (0.20, 0.30, 0.44, 0.70 give 0.28, 0.34, 0.45, 0.72). The marginal likelihood then gives a **declared error of the kidney datum, 0.33 in log**, against 0.08 before. So of the $[R_K]$ SD of 1.1 in nb18, most was datum error, under this target. The target is from published dose SDs, which contain measurement error; a smaller true spread would mean a smaller $\tau$. Median 30.1 compares with the five patients' 25.4.

**Spread of $\phi$.** Kidneys 0 (the target is met by $\tau$). Liver 0.15. Spleen 0, and over-dispersed at any setting (0.47 against 0.25, a single source with 5 patients). **Glands: the target does not confirm 0.3.** With posterior draws, the gland density at its prior already gives 0.51 against the target 0.46, so the chosen spread is 0, and 0.3 would give 0.58. That is the difference from §12, where point estimates gave 0.16 and made 0.3 look right.

**Whole body.** §13: no physically motivated fix, declared. Burden explains most of the shortfall, GFR is not the cause, and late excretion stays above Kurth 2018.

**Final values:** $\phi$ kidneys 0.31, glands 0.23, liver 2.10, spleen 0.85; $\lambda_{rel}$ kidneys 1.84e-4, liver and spleen 3.00e-4, glands 2.72e-4, lesions 5.8e-5 /min (log SD 0.18, 0.34, 0.15, 0.35; lesions plus a per-patient median of log SD 0.52); lesion median 181. Control: 2.21e-4, 2.79e-4, 2.83e-4, 6.4e-5, 227. Point values are for the cohort; their uncertainty is that of §12, which was not recomputed with these spreads.

## 17. Therapy amount from a measurement, and the lesion median chosen on the targets

**Therapy amount.** Wang et al., J Nucl Med 2025;66:824, measured PSMA-617 in 8 unused clinical Pluvicto doses by radio-HPLC: **6.31 ± 0.72 µg/mL** (n = 32, 4 samples per vial), with the assay of 1000 MBq/mL and injected volumes of 7.5-12.5 mL from the package insert, molar activity 100-212 GBq/µmol. The letter gives no per-dose mass, so it is derived here: concentration × volume, 1042.1 g/mol (ours, not in the letter), **about 45-76 nmol at the mean concentration, median about 60**. Each patient draws its concentration and its volume, uniform; the volume range stands in for how much decay has elapsed at injection. It replaces the assumed 100 nmol. **Limits:** 8 doses from one centre; supplement not read; older literature quotes 125-160 µg (about 120-154 nmol, cited by that letter, not read); and the published doses used as targets come partly from other preparations, whose mass we do not know.

**Lesion median, chosen on the targets**, since the PET does not identify it (§10). Grid on median and on a scale of its SDs at the measured amount, score $\sum z^2$ on four targets: Spearman of dose with TLV for kidneys, -0.43, and glands, -0.41, SE 0.15 from n = 30; lesion Gy/GBq at TLV at least 100 mL, 1.66, tolerance 0.3 in log; lesion dose spread, 0.78, tolerance 0.2. All on phantoms with TLV at least 100 mL except the spread. Then $\phi$ and $\lambda_{rel}$ recalibrated, median fixed, with lesion dose at high burden added to the targets. Grid on 180 phantoms × 2 draws of the first cohort.

In [ ]:
V1 = ROOT / "data/processed/psma/cohort_v1"
from scipy.stats import spearmanr
from dataclasses import replace as dc_replace
v1_bundles = [b for b in pickle.loads((V1 / "bundles_train.pkl").read_bytes()) if "error" not in b]
amounts_measured = np.array([C.measured_amount(b.seed) for b in Co.calibration_bases(v1_bundles, FINAL_SETTINGS["priors"], per_patient=2)])
print("Therapy amount, nmol: median", round(np.median(amounts_measured), 1), "| 5-95%", np.round(np.quantile(amounts_measured, [0.05, 0.95]), 1), "| log SD", round(np.log(amounts_measured).std(), 3))
BASES2 = Co.calibration_bases(v1_bundles, FINAL_SETTINGS["priors"], per_patient=2)
TLV2 = np.array([b.geometry.lesions_ml.sum() for b in BASES2])
HIGH2 = TLV2 >= 100.0
LESION_HIGH = float(np.exp(np.mean(np.log([1.84, 1.68, 11.55 / 7.8]))))
print(len(BASES2), "simulations per setting;", int(HIGH2.sum()), "with TLV at least 100 mL")


def job2(job):
    i, theta, phi_sd, tau, nmol = job
    return Co.forward_spread(BASES2[i], theta, phi_sd, REL_SD, tau, nmol)


def metrics2(rows):
    d = pd.DataFrame(rows).assign(tlv=TLV2)
    high = d[HIGH2]
    return dict(rho_kidneys=spearmanr(high.dose_kidneys, high.tlv)[0], rho_glands=spearmanr(high.dose_salivary, high.tlv)[0], rho_kidneys_all=spearmanr(d.dose_kidneys, d.tlv)[0], rho_glands_all=spearmanr(d.dose_salivary, d.tlv)[0], lesion_high=high.dose_lesions.mean(), lesion_all=d.dose_lesions.mean(), lesion_spread=float(np.log(d.dose_lesions.dropna()).std()), kidneys=d.dose_kidneys.mean(), glands=d.dose_salivary.mean(), sites_high=d.tumour_sites_nmol[HIGH2].median())


def score(m):
    return ((m["rho_kidneys"] + 0.43) / 0.153) ** 2 + ((m["rho_glands"] + 0.41) / 0.153) ** 2 + (np.log(m["lesion_high"] / LESION_HIGH) / 0.3) ** 2 + ((m["lesion_spread"] - SPREAD["dose_lesions"]) / 0.2) ** 2


def grid2_build():
    settings = [(m, t, "measured") for m in (200.0, 250.0, 300.0, 400.0, 500.0, 650.0, 800.0, 1100.0, 1600.0) for t in (0.85, 1.0, 1.15)] + [(400.0, 1.0, 100.0), (181.0, 1.0, 100.0), (181.0, 1.0, "measured")]
    out = []
    with mp.get_context("fork").Pool(N_PROCESSES) as workers:
        for m, t, a in settings:
            theta = theta_main.copy()
            theta[8] = np.log(m)
            rows = workers.map(job2, [(i, tuple(theta), tuple(PHI_LIST), t, a) for i in range(len(BASES2))], chunksize=12)
            out.append(dict(median=m, tau_scale=t, nmol=str(a), **metrics2(rows)))
    return out


grid2 = pd.DataFrame(cached("grid_measured", grid2_build))
grid2["score"] = [score(r) for _, r in grid2.iterrows()]
grid2.to_csv(OUT / "nb19_grid_measured_amount.csv", index=False)
display(grid2.sort_values("score").round(3))
best = grid2[grid2.nmol == "measured"].sort_values("score").iloc[0]
print("Chosen:", best["median"], best.tau_scale)
MEDIAN2, TAU2 = float(best["median"]), float(best.tau_scale)

In [ ]:
def recalibrate2(start, free, phi_sd):
    start = start.copy()
    start[8] = np.log(MEDIAN2)
    with mp.get_context("fork").Pool(N_PROCESSES) as workers:
        def residual(x):
            theta = start.copy()
            theta[free] = x
            rows = workers.map(job2, [(i, tuple(theta), tuple(phi_sd), TAU2, "measured") for i in range(len(BASES2))], chunksize=12)
            pop = C.population(rows)
            y = np.array([np.log(pop[n]) for n in C.OUTPUTS])
            high = np.mean([r["dose_lesions"] for r, h in zip(rows, HIGH2) if h and "dose_lesions" in r])
            return np.r_[((y - YA) / TA)[USED], (theta[7] - PRIOR_LESION[0]) / PRIOR_LESION[1], np.log(high / LESION_HIGH) / 0.3]
        solution = least_squares(residual, start[free], bounds=(LOW_A[free], HIGH_A[free]), diff_step=0.03, xtol=2e-3, ftol=2e-3, max_nfev=10)
    theta = start.copy()
    theta[free] = solution.x
    return theta


theta_main2 = cached("theta_main2", lambda: recalibrate2(theta_main, np.arange(8), PHI_LIST))
theta_control2 = cached("theta_control2", lambda: recalibrate2(theta_control, np.arange(4, 8), NONE))


def rows2_build(theta, phi_sd):
    with mp.get_context("fork").Pool(N_PROCESSES) as workers:
        return workers.map(job2, [(i, tuple(theta), tuple(phi_sd), TAU2, "measured") for i in range(len(BASES2))], chunksize=12)


rows_main2 = cached("rows_main2", lambda: rows2_build(theta_main2, PHI_LIST))
rows_control2 = cached("rows_control2", lambda: rows2_build(theta_control2, NONE))
table2 = pd.DataFrame(dict(main=np.exp(theta_main2), control=np.exp(theta_control2), first_cohort=np.exp(theta_main)), index=C.AGENT_NAMES)
table2.to_csv(OUT / "nb19_population_v2.csv")
display(table2.map(lambda v: float(f"{v:.3g}")))
display(pd.DataFrame({"target": agent_targets.value, "main": pd.Series(C.population(rows_main2)), "control": pd.Series(C.population(rows_control2))}).assign(main_ratio=lambda d: d.main / d.target, control_ratio=lambda d: d.control / d.target).round(3))
display(pd.DataFrame({"main": metrics2(rows_main2), "control": metrics2(rows_control2)}).round(3))
print("Score, main", round(score(metrics2(rows_main2)), 1), "| target lesion at high burden", round(LESION_HIGH, 2))
without2 = theta_main2.copy()
without2[:4] = 0.0
rows_without2 = cached("rows_without2", lambda: rows2_build(without2, NONE))
print("Kidney PET over therapy uptake at the same lambda_rel:", round(np.mean([r["dose_kidneys"] for r in rows_without2]) / np.mean([r["dose_kidneys"] for r in rows_main2]), 2), "| Peters 2.21")
(OUT / "nb19_cohort_population_v2.pkl").write_bytes(pickle.dumps(dict(main=dict(theta=theta_main2, phi_sd=PHI_SD), control=dict(theta=theta_control2, phi_sd=dict(zip(C.AGENT_ORGANS, NONE))), rel_sd=dict(zip(("kidneys", "liver", "salivary", "lesions"), REL_SD)), lesion_epistemic_sd=LESION_EPISTEMIC, settings=FINAL_SETTINGS, hyper_median=float(np.log(MEDIAN2)), hyper_median_control=float(np.log(MEDIAN2)), tau_scale=TAU2, tau=TAU, spread_targets=SPREAD, targets=agent_targets, lesion_high_target=LESION_HIGH)))

## 18. Report of §17

- **Amount:** derived from Wang 2025, median 61 nmol, 5-95% 45-80 in this sample, log SD 0.18.
- **Choice:** median 800, SD × 1.15, best score of 27 settings at the measured amount. It is 2.9 prior SD above the start, and is a calibration to doses, not an estimate.
- **Final, main:** $\phi$ kidneys 0.28, glands 0.19, liver 1.77, spleen 0.71; $\lambda_{rel}$ 1.88e-4, 3.10e-4, 2.73e-4, 1.05e-4 /min. Kidney PET over therapy uptake 3.4, Peters 2.2: same sign.
- **The burden correlations of this section were optimistic** (-0.25, -0.17 on 94 simulations): the generated cohort gives -0.07, -0.06 (nb20 §8). Selection on noise, with a smaller part from the refit under the higher median.
- **Population for the cohort:** `nb19_cohort_population_v2.pkl`. The earlier `nb19_cohort_population.pkl` is that of the first cohort, kept in `cohort_v1/`.